# Praktikum Penambangan Data (PPD) - Pertemuan 6
## Topik: Clustering dan Association Rule
- **Nama**: Hafidz Rizqullah Prasetya
- **NIM**: 24/535493/SV/24243
- **Kelas**: PL5A1
- **Program Studi**: D-IV Teknologi Rekayasa Perangkat Lunak
- **Departemen**: Teknik Elektro dan Informatika, Sekolah Vokasi, Universitas Gadjah Mada

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hafidzrizqullahprasetya/PPD/blob/main/pertemuan-6/praktikum-6-clustering.ipynb)


---
# Bagian 1: Percobaan Latihan Praktikum (Credit Card Dataset)
Percobaan ini mengimplementasikan segmentasi nasabah pemegang kartu kredit memakai dataset `CC GENERAL.csv`. Tahapan meliputi eksplorasi data, pembersihan, imputasi nilai hilang, standarisasi fitur, penentuan K optimal (Elbow Method & Silhouette Score), pemodelan K-Means dan K-Medoids, visualisasi 2D dan 3D, serta analisis profil klaster nasabah.


### 1.1 Instalasi Library Tambahan
Instalasi pustaka `scikit-learn-extra` (untuk K-Medoids) dan `kneed` (untuk deteksi titik siku otomatis).


In [ ]:
!pip install scikit-learn-extra kneed


### 1.2 Impor Library
Mengimpor modul-modul yang dibutuhkan sepanjang praktikum.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn_extra.cluster import KMedoids
from kneed import KneeLocator

plt.style.use('ggplot')
%matplotlib inline


### 1.3 Membaca dan Memuat Dataset
Dataset dibaca dari repositori mirror data science practice `CC GENERAL.csv`.


In [ ]:
url_cc = 'https://raw.githubusercontent.com/ganjar87/data_science_practice/main/CC%20GENERAL.csv'
df = pd.read_csv(url_cc)
print("Dimensi dataset:", df.shape)
df.head()


### 1.4 Inspeksi Struktur dan Ringkasan Statistik Data


In [ ]:
df.info()


In [ ]:
df.describe().T


### 1.5 Pemeriksaan Korelasi Antar-Variabel Numerik


In [ ]:
df.corr(numeric_only=True)


### 1.6 Pembersihan Data (Data Cleaning)
Kolom identitas nasabah `CUST_ID` dihapus karena tidak memiliki arti geometris dalam ruang fitur.


In [ ]:
df_new = df.drop(['CUST_ID'], axis=1)
print("Dimensi setelah drop CUST_ID:", df_new.shape)
df_new.head()


### 1.7 Identifikasi Nilai Hilang (Missing Values)


In [ ]:
df_new.isnull().sum()


### 1.8 Imputasi Nilai Hilang dengan Nilai Median
Fitur `MINIMUM_PAYMENTS` dan `CREDIT_LIMIT` memiliki nilai kosong. Imputasi menggunakan median dipilih agar tidak terpengaruh oleh pencilan.


In [ ]:
df_new['MINIMUM_PAYMENTS'] = df_new['MINIMUM_PAYMENTS'].fillna(df_new['MINIMUM_PAYMENTS'].median())
df_new['CREDIT_LIMIT'] = df_new['CREDIT_LIMIT'].fillna(df_new['CREDIT_LIMIT'].median())

print("Pengecekan ulang missing value:")
print(df_new.isnull().sum())


### 1.9 Standarisasi Skala Fitur (StandardScaler)
Algoritma klasterisasi sensitif terhadap rentang skala. Seluruh fitur distandarisasi agar berdistribusi rata-rata nol dan standar deviasi satu.


In [ ]:
X = df_new.astype(float).values
scaler = StandardScaler().fit(X)
X_new = scaler.transform(X)
print("Bentuk matriks ternormalisasi:", X_new.shape)
X_new


### 1.10 Penentuan Nilai K Optimal: Elbow Method pada K-Means
Menguji nilai K = 1 sampai 10 dan mencatat nilai inertia (Within-Cluster Sum of Squares / WCSS).


In [ ]:
inertia_list = []
for num_clusters in range(1, 11):
    kmeans_model = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    kmeans_model.fit(X_new)
    inertia_list.append(kmeans_model.inertia_)
    print(f"For n_clusters = {num_clusters}, inertia value is {kmeans_model.inertia_:.2f}")


### 1.11 Visualisasi Kurva Elbow K-Means dan Deteksi Titik Siku dengan KneeLocator


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(range(1, 11), inertia_list, marker='o', linewidth=2, markersize=8)
plt.xlabel("Number of Clusters (K)", size=13)
plt.ylabel("Inertia Value (WCSS)", size=13)
plt.title("Elbow Method: Nilai Inertia terhadap Jumlah Klaster (K-Means)")
plt.show()

kneedle = KneeLocator(range(1, 11), inertia_list, S=1.0, curve='convex', direction='decreasing')
print("Optimal Knee Point:", round(kneedle.knee, 3))
print("Optimal Elbow Point:", round(kneedle.elbow, 3))

kneedle.plot_knee()
plt.title("Validasi Titik Siku Kurva K-Means (KneeLocator)")
plt.show()


### 1.12 Penentuan Nilai K Optimal: Elbow Method pada K-Medoids
Menguji variasi nilai K = 1 sampai 10 memakai algoritma K-Medoids (PAM).


In [ ]:
inertia_list_kmed = []
for num_clusters in range(1, 11):
    kmedoids_model = KMedoids(n_clusters=num_clusters, random_state=42)
    kmedoids_model.fit(X_new)
    inertia_list_kmed.append(kmedoids_model.inertia_)
    print(f"The inertia of {num_clusters} clusters : {kmedoids_model.inertia_:.2f}")


### 1.13 Visualisasi Kurva Elbow K-Medoids dan Deteksi KneeLocator


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(range(1, 11), inertia_list_kmed, marker='o', linewidth=2, markersize=8, color='darkorange')
plt.xlabel("Number of Clusters (K)", size=13)
plt.ylabel("Inertia Value", size=13)
plt.title("Elbow Method: Nilai Inertia terhadap Jumlah Klaster (K-Medoids)")
plt.show()

kneedle_med = KneeLocator(range(1, 11), inertia_list_kmed, S=1.0, curve='convex', direction='decreasing')
print("Optimal Knee Point (K-Medoids):", kneedle_med.knee)

kneedle_med.plot_knee()
plt.title("Validasi Titik Siku Kurva K-Medoids (KneeLocator)")
plt.show()


### 1.14 Evaluasi Silhouette Score pada K-Means
Menghitung koefisien siluet untuk rentang K = 2 sampai 10.


In [ ]:
sh_list = []
for num_clusters in range(2, 11):
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_new)
    score = silhouette_score(X_new, cluster_labels)
    sh_list.append(score)
    print(f"For n_clusters = {num_clusters}, silhouette score is {score:.4f}")

plt.figure(figsize=(8, 6))
plt.plot(range(2, 11), sh_list, marker='o', linewidth=2, markersize=8, color='royalblue')
plt.xlabel("Number of Clusters (K)", size=13)
plt.ylabel("Silhouette Score", size=13)
plt.title("Evaluasi Silhouette Score terhadap Jumlah Klaster (K-Means)")
plt.show()


### 1.15 Evaluasi Silhouette Score pada K-Medoids


In [ ]:
sh_list_kmed = []
for num_clusters in range(2, 11):
    kmedoids = KMedoids(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmedoids.fit_predict(X_new)
    score = silhouette_score(X_new, cluster_labels)
    sh_list_kmed.append(score)
    print(f"For n_clusters = {num_clusters}, silhouette score is {score:.4f}")

plt.figure(figsize=(8, 6))
plt.plot(range(2, 11), sh_list_kmed, marker='o', linewidth=2, markersize=8, color='crimson')
plt.xlabel("Number of Clusters (K)", size=13)
plt.ylabel("Silhouette Score", size=13)
plt.title("Evaluasi Silhouette Score terhadap Jumlah Klaster (K-Medoids)")
plt.show()


### 1.16 Pelatihan Model Definitif K-Means (K = 4)
Setelah analisis Elbow Method dan pertimbangan segmentasi bisnis mengonfirmasi K = 4, model K-Means dilatih secara penuh.


In [ ]:
k_means = KMeans(n_clusters=4, random_state=42, n_init=10)
k_means.fit(X_new)
labels = k_means.labels_
df_new['cluster_labels'] = labels

print("Koordinat Centroids K-Means (ruang 17 dimensi ternormalisasi):")
centroids = k_means.cluster_centers_
print(centroids)
df_new.head()


### 1.17 Visualisasi Sebaran Klaster 2D K-Means (Matplotlib & Seaborn)


In [ ]:
x1 = df_new['PURCHASES']
x2 = df_new['PAYMENTS']

plt.figure(figsize=(8, 6))
u_labels = np.unique(labels)
for i in u_labels:
    mask = df_new['cluster_labels'] == i
    plt.scatter(x1[mask], x2[mask], label=f'Cluster {i}', alpha=0.6)
plt.xlabel(x1.name, fontsize=13)
plt.ylabel(x2.name, fontsize=13)
plt.title('K-Means Clustering: PURCHASES vs PAYMENTS (Matplotlib)', fontsize=14)
plt.legend()
plt.show()

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PURCHASES', y='PAYMENTS', hue='cluster_labels', data=df_new, palette='Paired', alpha=0.7)
plt.legend(loc='lower right')
plt.title('K-Means Clustering Scatter Plot (Seaborn)', fontsize=14)
plt.show()


### 1.18 Visualisasi Sebaran Klaster 3D Interaktif (Plotly Express)


In [ ]:
fig = px.scatter_3d(df_new, x='PURCHASES', y='PAYMENTS', z='BALANCE', 
                    color=df_new['cluster_labels'].astype(str), 
                    labels={'color': 'Klaster'},
                    title='Visualisasi 3D K-Means: PURCHASES vs PAYMENTS vs BALANCE')
fig.show()


### 1.19 Pelatihan Model Definitif K-Medoids (K = 4) dan Medoids


In [ ]:
k_medoids = KMedoids(n_clusters=4, random_state=42)
k_medoids.fit(X_new)
labels_med = k_medoids.labels_
df_new['cluster_labels_kmed'] = labels_med

print("Koordinat Medoids (titik riil dataset ternormalisasi):")
medoid_centers = k_medoids.cluster_centers_
print(medoid_centers)
df_new.head()


### 1.20 Visualisasi Sebaran Klaster 2D K-Medoids


In [ ]:
plt.figure(figsize=(8, 6))
for i in np.unique(labels_med):
    mask = df_new['cluster_labels_kmed'] == i
    plt.scatter(x1[mask], x2[mask], label=f'Cluster {i}', alpha=0.6)
plt.xlabel('PURCHASES', fontsize=13)
plt.ylabel('PAYMENTS', fontsize=13)
plt.title('K-Medoids Clustering: PURCHASES vs PAYMENTS (Matplotlib)', fontsize=14)
plt.legend()
plt.show()

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PURCHASES', y='PAYMENTS', hue='cluster_labels_kmed', data=df_new, palette='Set2', alpha=0.7)
plt.legend(loc='lower right')
plt.title('K-Medoids Clustering Scatter Plot (Seaborn)', fontsize=14)
plt.show()


### 1.21 Analisis Profil Klaster (Cluster Profiling)
Agregasi nilai rata-rata dan total per klaster untuk menginterpretasikan makna finansial tiap segmen nasabah.


In [ ]:
df_out_mean = df_new.groupby('cluster_labels').mean(numeric_only=True).reset_index()
print("Tabel Rata-rata Fitur Finansial per Klaster:")
df_out_mean[['cluster_labels', 'BALANCE', 'PURCHASES', 'PAYMENTS', 'CASH_ADVANCE', 'CREDIT_LIMIT']]


In [ ]:
df_out_sum = df_new.groupby('cluster_labels')[['PURCHASES', 'PAYMENTS', 'BALANCE']].sum().reset_index()
print("Tabel Akumulasi Total Fitur Finansial per Klaster:")
df_out_sum


In [ ]:
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
sns.barplot(x='cluster_labels', y='PURCHASES', data=df_out_mean, palette='Blues_d')
plt.title('Rata-rata PURCHASES')
plt.xlabel('Klaster')

plt.subplot(1, 3, 2)
sns.barplot(x='cluster_labels', y='PAYMENTS', data=df_out_mean, palette='Greens_d')
plt.title('Rata-rata PAYMENTS')
plt.xlabel('Klaster')

plt.subplot(1, 3, 3)
sns.barplot(x='cluster_labels', y='BALANCE', data=df_out_mean, palette='Reds_d')
plt.title('Rata-rata BALANCE')
plt.xlabel('Klaster')

plt.tight_layout()
plt.show()


### 1.22 Interpretasi Bisnis dan Rekomendasi Strategi Perbankan
1. **Cluster 0 (Nasabah Inaktif / Low Engagement):**
   - *Profil*: Saldo, transaksi belanja, dan pembayaran berada pada tingkat terendah.
   - *Strategi*: Program bebas iuran tahunan (annual fee waiver), promo diskon transaksi pertama, aktivasi notifikasi berkala.
2. **Cluster 1 (Nasabah Transaksi Rutin / Moderate Spenders):**
   - *Profil*: Transaksi belanja, saldo, dan pembayaran berada pada tingkat menengah proporsional.
   - *Strategi*: Program poin loyalitas (reward points), opsi cicilan 0% kebutuhan belanja bulanan, penawaran cashback kategori supermarket.
3. **Cluster 2 (Nasabah Belanja Tinggi / VIP Prime Transactors):**
   - *Profil*: Akumulasi belanja dan pembayaran tagihan sangat tinggi dengan limit besar. Nasabah berdisiplin finansial tinggi.
   - *Strategi*: Peningkatan batas kredit (credit limit increase), undangan peningkatan ke tier Platinum/World Elite, layanan prioritas airport lounge.
4. **Cluster 3 (Nasabah Saldo Tinggi & Penarikan Tunai / Revolvers):**
   - *Profil*: Saldo utang bergulir dan penarikan tunai ATM paling tinggi. Menghasilkan marjin bunga besar bagi bank, tetapi memiliki risiko kredit lebih tinggi.
   - *Strategi*: Program restrukturisasi saldo menjadi cicilan tetap berbunga kompetitif untuk memitigasi risiko gagal bayar (default risk).


---
# Bagian 2: Tugas dan Analisis Praktikum

## 2.1 Soal 1: Dataset Preparation - Credit Card Dataset
- **Tautan Unduhan**: https://www.kaggle.com/datasets/arjunbhasin2013/ccdata
- **Tujuan Penggunaan Dataset**:
  Dataset ini digunakan untuk membangun model segmentasi nasabah kartu kredit berbasis perilaku konsumsi dan pembayaran selama enam bulan. Pemahaman kelompok nasabah memungkinkan manajemen perbankan merancang strategi pemasaran produk yang dipersonalisasi (*targeted marketing*), mengelola alokasi limit kredit berdasarkan profil risiko (*credit risk management*), serta meningkatkan loyalitas nasabah guna mencegah churn.

### Tabel Deskripsi Lengkap 18 Fitur Dataset
| No | Nama Kolom | Tipe Data | Deskripsi dan Makna Operasional |
| :--- | :--- | :--- | :--- |
| 1 | `CUST_ID` | String | Nomor identitas unik pemegang kartu kredit (dihapus saat pemodelan karena bersifat kategorikal murni). |
| 2 | `BALANCE` | Float | Jumlah saldo utang yang masih tersisa pada rekening kartu kredit nasabah yang belum dilunasi. |
| 3 | `BALANCE_FREQUENCY` | Float | Rasio seberapa sering saldo diperbarui, dengan skala skor 0 (tidak pernah) hingga 1 (selalu diperbarui berkala). |
| 4 | `PURCHASES` | Float | Total akumulasi nominal pembelian yang dilakukan nasabah selama 6 bulan. |
| 5 | `ONEOFF_PURCHASES` | Float | Total nominal transaksi pembelian yang dibayar tunai sekaligus tanpa menggunakan skema cicilan. |
| 6 | `INSTALLMENTS_PURCHASES` | Float | Total nominal transaksi belanja yang dibayar melalui skema cicilan bertahap. |
| 7 | `CASH_ADVANCE` | Float | Total nominal penarikan dana tunai di muka melalui mesin ATM. |
| 8 | `PURCHASES_FREQUENCY` | Float | Frekuensi transaksi pembelian yang dilakukan nasabah (rasio skor 0 sampai 1). |
| 9 | `ONEOFF_PURCHASES_FREQUENCY` | Float | Frekuensi transaksi belanja tunai sekaligus (rasio skor 0 sampai 1). |
| 10 | `PURCHASES_INSTALLMENTS_FREQUENCY` | Float | Frekuensi transaksi belanja menggunakan cicilan bertahap (rasio 0 sampai 1). |
| 11 | `CASH_ADVANCE_FREQUENCY` | Float | Frekuensi nasabah melakukan transaksi penarikan tunai di muka (rasio 0 sampai 1). |
| 12 | `CASH_ADVANCE_TRX` | Integer | Total jumlah transaksi penarikan uang tunai di muka yang berhasil dilakukan. |
| 13 | `PURCHASES_TRX` | Integer | Total jumlah transaksi pembelian yang dilakukan oleh nasabah. |
| 14 | `CREDIT_LIMIT` | Float | Batas pagu kredit maksimum yang dialokasikan bank kepada pemegang kartu. |
| 15 | `PAYMENTS` | Float | Total nominal pembayaran tagihan yang telah disetorkan oleh nasabah. |
| 16 | `MINIMUM_PAYMENTS` | Float | Total nominal pembayaran minimum yang telah dibayarkan nasabah untuk menjaga kartu tetap aktif. |
| 17 | `PRC_FULL_PAYMENT` | Float | Persentase pembayaran tagihan yang dilunasi penuh (100%) oleh nasabah setiap bulan. |
| 18 | `TENURE` | Integer | Masa aktif kepemilikan kartu kredit oleh nasabah (dalam satuan bulan, rentang 6 hingga 12 bulan). |


## 2.2 Soal 2: Eksperimen Klasterisasi Dataset Tambahan (Air Traffic Passenger Statistics)
- **Tautan Resmi**: https://data.sfgov.org/Transportation/Air-Traffic-Passenger-Statistics/rkru-6vcg/about_data
- **Deskripsi**: Dataset mencatat statistik operasional lalu lintas penumpang bulanan di Bandara Internasional San Francisco (SFO) dari portal resmi DataSF Pemerintah Kota San Francisco.
- **Tujuan**: Membangun model segmentasi maskapai dan pola operasional penumpang memakai K-Means dan K-Medoids, serta menguji ketahanan kedua algoritma terhadap pencilan (*outliers*) operasional bandara.


In [ ]:
url_air = "https://data.sfgov.org/api/views/rkru-6vcg/rows.csv?accessType=DOWNLOAD"
try:
    df_air = pd.read_csv(url_air)
    print("Berhasil mengunduh dataset SFO dari portal resmi DataSF!")
except Exception:
    df_air = pd.read_csv('Air_Traffic_Passenger_Statistics.csv')

print("Dimensi dataset SFO:", df_air.shape)
df_air.head()


### 2.2.1 Eksplorasi Struktur dan Tipe Data Dataset Penerbangan


In [ ]:
df_air.info()


In [ ]:
df_air.describe().T


In [ ]:
print("Jumlah missing values per kolom:")
print(df_air.isnull().sum())


### 2.2.2 Preprocessing: Seleksi Fitur dan Label Encoding Variabel Kategorikal
Fitur non-numerik seperti identitas maskapai, kategori geografis, jenis aktivitas penerbangan, kategori tarif, terminal, dan area keberangkatan ditransformasikan menggunakan `LabelEncoder` dari `sklearn.preprocessing`.


In [ ]:
from sklearn.preprocessing import LabelEncoder

act_col = 'Activity Type Code' if 'Activity Type Code' in df_air.columns else 'Activity Type Description'
features_air = ['Activity Period', 'Operating Airline', 'GEO Summary', 'GEO Region', 
                act_col, 'Price Category Code', 'Terminal', 'Boarding Area', 'Passenger Count']
df_air_sub = df_air[[c for c in features_air if c in df_air.columns]].dropna().copy()

cats = df_air_sub.select_dtypes(include=['object', 'bool', 'string']).columns
cat_features = list(cats.values)

le = LabelEncoder()
for col in cat_features:
    df_air_sub[col] = le.fit_transform(df_air_sub[col].astype(str))

print("DataFrame pasca-label encoding:")
df_air_sub.head()


### 2.2.3 Standarisasi Fitur Menggunakan StandardScaler


In [ ]:
X_air = df_air_sub.values.astype(float)
scaler_air = StandardScaler()
X_air_scaled = scaler_air.fit_transform(X_air)
print("Bentuk matriks fitur ternormalisasi:", X_air_scaled.shape)


### 2.2.4 Penentuan Nilai K Optimal pada Data Penerbangan (Elbow Method & Silhouette Score)
Menguji rentang nilai K = 1 sampai 10 menggunakan K-Means.


In [ ]:
inertia_air = []
sil_air = []

sample_size = min(3000, len(X_air_scaled))
np.random.seed(42)
sample_idx = np.random.choice(len(X_air_scaled), size=sample_size, replace=False)
X_air_sample = X_air_scaled[sample_idx]

for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_air_sample)
    inertia_air.append(km.inertia_)
    if k >= 2:
        score = silhouette_score(X_air_sample, km.labels_)
        sil_air.append(score)
        print(f"K = {k}: Inertia = {km.inertia_:.2f}, Silhouette = {score:.4f}")
    else:
        print(f"K = {k}: Inertia = {km.inertia_:.2f}")


In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, 11), inertia_air, marker='o', linewidth=2, markersize=8, color='navy')
plt.xlabel("Jumlah Klaster (K)", size=12)
plt.ylabel("Inertia (WCSS)", size=12)
plt.title("Elbow Method Data Penerbangan SFO")

plt.subplot(1, 2, 2)
plt.plot(range(2, 11), sil_air, marker='s', linewidth=2, markersize=8, color='darkgreen')
plt.xlabel("Jumlah Klaster (K)", size=12)
plt.ylabel("Silhouette Score", size=12)
plt.title("Silhouette Score Data Penerbangan SFO")

plt.tight_layout()
plt.show()

kneedle_air = KneeLocator(range(1, 11), inertia_air, curve='convex', direction='decreasing')
print("Optimal Knee Point (Data Penerbangan):", kneedle_air.knee)


### 2.2.5 Penerapan Model K-Means dan K-Medoids pada Data Penerbangan (K = 4)


In [ ]:
kmeans_air = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_air_km = kmeans_air.fit_predict(X_air_sample)

kmedoids_air = KMedoids(n_clusters=4, random_state=42)
labels_air_kmed = kmedoids_air.fit_predict(X_air_sample)

df_eval_air = df_air_sub.iloc[sample_idx].copy()
df_eval_air['cluster_kmeans'] = labels_air_km
df_eval_air['cluster_kmedoids'] = labels_air_kmed
df_eval_air.head()


### 2.2.6 Visualisasi Perbandingan Sebaran Klaster K-Means vs K-Medoids


In [ ]:
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
sns.scatterplot(x='Operating Airline', y='Passenger Count', hue='cluster_kmeans', 
                data=df_eval_air, palette='tab10', alpha=0.7)
plt.title('Segmentasi Penerbangan SFO: K-Means (K=4)')
plt.xlabel('Kode Maskapai (Encoded)')
plt.ylabel('Jumlah Penumpang (Passenger Count)')

plt.subplot(1, 2, 2)
sns.scatterplot(x='Operating Airline', y='Passenger Count', hue='cluster_kmedoids', 
                data=df_eval_air, palette='tab10', alpha=0.7)
plt.title('Segmentasi Penerbangan SFO: K-Medoids (K=4)')
plt.xlabel('Kode Maskapai (Encoded)')
plt.ylabel('Jumlah Penumpang (Passenger Count)')

plt.tight_layout()
plt.show()


### 2.2.7 Analisis Karakteristik Segmen Maskapai yang Terbentuk
1. **Segmen 1 (Maskapai Hub Domestik Volume Sangat Tinggi):** Operator utama bandara SFO (United Airlines dan SkyWest) dengan rute domestik dan ratusan ribu penumpang per bulan.
2. **Segmen 2 (Maskapai Bertarif Rendah / LCC):** Maskapai penerbangan bertarif hemat (Southwest, JetBlue) yang melayani rute domestik dan regional dengan volume sedang.
3. **Segmen 3 (Maskapai Internasional Jarak Jauh):** Maskapai penerbangan lintas benua berlayanan penuh (Lufthansa, British Airways, Singapore Airlines, Cathay Pacific) dengan terminal keberangkatan internasional dan volume terjadwal reguler.
4. **Segmen 4 (Penerbangan Regional, Musiman, dan Khusus):** Penerbangan musiman atau rute regional dengan frekuensi terbatas dan volume penumpang kecil.


### 2.2.8 Analisis Ketahanan terhadap Pencilan: K-Means vs K-Medoids
- **Karakteristik Pencilan pada Data Penerbangan:**
  Bandara SFO memiliki ketimpangan lalu lintas alami yang tajam karena United Airlines bertindak sebagai maskapai hub dengan pangsa lebih dari 40% dari total operasional. Jumlah penumpang United Airlines dapat mencapai jutaan orang per bulan, sementara sebagian besar maskapai internasional hanya melayani puluhan ribu penumpang.
- **Kerentanan Centroid K-Means:**
  K-Means menghitung centroid sebagai rata-rata aritmetika (*mean*). Karena galat jarak dikuadratkan dalam fungsi objektif WCSS, titik pencilan bervolume jutaan penumpang memberikan penalti kuadratik yang besar. Akibatnya, centroid tertarik menjauh ke arah nilai ekstrim, mendistorsi batas partisi kelompok mayoritas maskapai normal.
- **Ketahanan Medoid K-Medoids:**
  K-Medoids memilih objek data aktual di dalam dataset sebagai pusat kelompok (medoid) dan meminimalkan jumlah jarak absolut Manhattan/Euclidean. Menukar medoid ke titik pencilan ekstrem akan memperbesar total selisih jarak absolut terhadap semua titik data lainnya, sehingga biaya pertukaran S >= 0. Pertukaran tersebut otomatis ditolak, menjaga medoid tetap stabil mewakili mayoritas data normal.


---
# Bagian 3: Association Rule Mining & Algoritma Apriori (Materi Pendukung Modul 6)
Bagian ini mendemonstrasikan perhitungan matematis aturan asosiasi keranjang belanja (*Market Basket Analysis*), metrik evaluasi (Support, Confidence, Lift), serta prinsip Apriori (*anti-monotonicity of support*).


In [ ]:
transaksi = [
    {'Bread', 'Milk'},
    {'Bread', 'Vegetables', 'Fruits', 'Eggs'},
    {'Milk', 'Vegetables', 'Fruits', 'Coke'},
    {'Bread', 'Milk', 'Vegetables', 'Fruits'},
    {'Bread', 'Milk', 'Vegetables', 'Coke'}
]

total_T = len(transaksi)
print(f"Total transaksi |T|: {total_T}")

X = {'Milk', 'Vegetables'}
Y = {'Fruits'}
XY = X.union(Y)

sigma_X = sum(1 for t in transaksi if X.issubset(t))
sigma_Y = sum(1 for t in transaksi if Y.issubset(t))
sigma_XY = sum(1 for t in transaksi if XY.issubset(t))

support_XY = sigma_XY / total_T
support_X = sigma_X / total_T
support_Y = sigma_Y / total_T

confidence = sigma_XY / sigma_X
lift = support_XY / (support_X * support_Y)

print(f"Sigma(X = {X}) = {sigma_X}")
print(f"Sigma(Y = {Y}) = {sigma_Y}")
print(f"Sigma(X U Y = {XY}) = {sigma_XY}")
print("-" * 40)
print(f"Support (s)    = {support_XY:.2f} ({support_XY*100:.1f}%)")
print(f"Confidence (c) = {confidence:.4f} ({confidence*100:.1f}%)")
print(f"Lift           = {lift:.2f}")
print("-" * 40)
if lift > 1:
    print("Interpretasi: Nilai Lift > 1 membuktikan adanya hubungan asosiasi positif yang valid antar-item.")
elif lift == 1:
    print("Interpretasi: Nilai Lift = 1 menandakan kejadian X dan Y saling independen.")
else:
    print("Interpretasi: Nilai Lift < 1 menandakan hubungan substitutif / saling meniadakan.")


### Rangkuman Pembuktian Prinsip Apriori
1. **Sifat Penurunan Monoton (Anti-monotonicity of Support):**
   Jika suatu itemset memenuhi ambang batas min_sup (frequent), maka setiap subset di dalamnya dijamin pasti frequent. Sebaliknya, jika suatu subset tidak sering (infrequent), maka seluruh superset yang memuatnya dipastikan tidak sering.
2. **Pemangkasan Kisi (Lattice Pruning):**
   Dengan prinsip ini, algoritma Apriori memangkas pohon kombinasi kandidat itemset tanpa harus membaca ulang database, menekan kompleksitas pencarian dari 2^d - 1 menjadi ruang kandidat yang jauh lebih ringkas dan efisien.
